# 4. RAG + LLM Obesity Guideline Recommendation

Pipeline overview:
1. Load guideline files from `data/guidelines/`
2. Split them into sentence-based chunks
3. Embed all chunks with SentenceTransformer
4. Use cluster summaries as retrieval queries
5. Send retrieved evidence and the profile summary to Gemini
6. Evaluate generated recommendations with BERTScore

In [ ]:
%pip install -q sentence-transformers PyPDF2 nltk bert-score google-generativeai python-dotenv

In [ ]:
import glob
import json
import os

import google.generativeai as genai
import nltk
import numpy as np
import PyPDF2
from dotenv import load_dotenv
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
load_dotenv('../.env')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY not found. Create .env from .env.example and set your key.')
genai.configure(api_key=GEMINI_API_KEY)
ARTIFACTS_DIR = '../artifacts'
GUIDELINES_DIR = '../data/guidelines'

## Load guideline files and build text chunks

In [ ]:
def load_text_from_file(path: str) -> str:
    if path.endswith('.pdf'):
        pages = []
        with open(path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    pages.append(text)
        return ' '.join(pages)
    with open(path, 'r', encoding='utf-8') as file:
        return file.read()

def make_chunks(text: str, window: int = 3, stride: int = 2) -> list[str]:
    sentences = [sentence.strip() for sentence in sent_tokenize(text) if len(sentence.strip()) > 20]
    chunks = []
    for index in range(0, len(sentences), stride):
        chunk = ' '.join(sentences[index:index + window])
        if chunk:
            chunks.append(chunk)
    return chunks

all_chunks = []
all_sources = []
files = glob.glob(f'{GUIDELINES_DIR}/*.txt') + glob.glob(f'{GUIDELINES_DIR}/*.pdf')
if not files:
    raise FileNotFoundError(f'No guideline files found in {GUIDELINES_DIR}')
for path in files:
    source_name = os.path.basename(path)
    text = load_text_from_file(path)
    chunks = make_chunks(text)
    all_chunks.extend(chunks)
    all_sources.extend([source_name] * len(chunks))
    print(f'{source_name}: {len(chunks)} chunks')
print(f'Total chunks: {len(all_chunks)}')

## Generate and save chunk embeddings

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embedding_model.encode(all_chunks, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
np.savez(f'{ARTIFACTS_DIR}/rag_chunks.npz', embeddings=chunk_embeddings, chunks=np.array(all_chunks), sources=np.array(all_sources))
print(f'Saved artifacts/rag_chunks.npz with shape {chunk_embeddings.shape}')

## Load cluster summaries

In [ ]:
with open(f'{ARTIFACTS_DIR}/cluster_summaries.json', encoding='utf-8') as file:
    cluster_summaries = json.load(file)
print(f'Loaded {len(cluster_summaries)} cluster summaries')
cluster_summaries[:2]

## Retrieval helper

In [ ]:
def retrieve_chunks(query: str, top_k: int = 5) -> list[dict]:
    stored = np.load(f'{ARTIFACTS_DIR}/rag_chunks.npz', allow_pickle=True)
    embeddings = stored['embeddings']
    chunks = stored['chunks'].tolist()
    sources = stored['sources'].tolist()
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{'chunk': chunks[index], 'source': sources[index], 'score': float(scores[index])} for index in top_indices]

## Gemini recommendation helper

In [ ]:
def generate_recommendation(cluster_summary: str, top_k: int = 5) -> dict:
    retrieved = retrieve_chunks(cluster_summary, top_k=top_k)
    context = '\n\n'.join(f"[Source: {item['source']}] {item['chunk']}" for item in retrieved)
    prompt = (
        'You are an expert obesity management clinician.\n'
        'Use the patient group profile and guideline excerpts below to write concise, evidence-based recommendations.\n\n'
        f'## Patient Group Profile\n{cluster_summary}\n\n'
        f'## Guideline Evidence\n{context}\n\n'
        'Respond with: (1) main risks, (2) 3-5 dietary actions, (3) 2-3 exercise actions, and (4) behavioral support ideas.'
    )
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(prompt)
    return {'summary': cluster_summary, 'retrieved': retrieved, 'recommendation': response.text}

test_cluster = cluster_summaries[0]
print(test_cluster['summary'])

In [ ]:
test_result = generate_recommendation(test_cluster['summary'], top_k=5)
print(test_result['recommendation'])
for item in test_result['retrieved']:
    print(item['score'], item['source'])

## Generate recommendations for all clusters

In [ ]:
import time

all_results = []
for cluster_summary in cluster_summaries:
    print(f"Processing cluster {cluster_summary['cluster']}")
    result = generate_recommendation(cluster_summary['summary'], top_k=5)
    all_results.append({
        'cluster': cluster_summary['cluster'],
        'label': cluster_summary['dominant_label'],
        'recommendation': result['recommendation'],
    })
    time.sleep(1)
print(f'Generated recommendations for {len(all_results)} clusters')

## Evaluate recommendation quality with BERTScore

In [ ]:
from bert_score import score as bert_score

if all_results:
    candidates = [item['recommendation'] for item in all_results]
    references = [retrieve_chunks(cluster_summaries[index]['summary'], top_k=1)[0]['chunk'] for index in range(len(all_results))]
    precision, recall, f1 = bert_score(candidates, references, lang='en', model_type='distilbert-base-uncased', verbose=False)
    print(f'BERTScore precision: {precision.mean():.4f}')
    print(f'BERTScore recall: {recall.mean():.4f}')
    print(f'BERTScore F1: {f1.mean():.4f}')
else:
    print('No generated results available. Run the previous cell first.')